In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS fraud_feature_store")

DataFrame[]

In [0]:
from pyspark.sql.functions import *

transactions = spark.table("fraud_bronze.bronze_transactions")

customer = spark.table("fraud_silver.silver_customer_master")

merchant = spark.table("fraud_silver.silver_merchant_master")

country = spark.table("fraud_silver.silver_country_risk")

exchange = spark.table("fraud_silver.silver_exchange_rates")

ofac = spark.table("fraud_silver.silver_ofac")

In [0]:
customer_features = (
    transactions
    .groupBy("card1")
    .agg(
        count("*").alias("total_transactions"),
        avg("TransactionAmt").alias("avg_transaction_amount"),
        max("TransactionAmt").alias("max_transaction_amount"),
        min("TransactionAmt").alias("min_transaction_amount"),
        expr("percentile(TransactionAmt,0.5)").alias("median_transaction_amount")
    )
    .withColumnRenamed("card1","customer_id")
)

In [0]:
spark.sql("""
DROP TABLE IF EXISTS fraud_feature_store.customer_features
""")


DataFrame[]

In [0]:
spark.sql("""
DROP TABLE IF EXISTS fraud_feature_store.merchant_features
""")

DataFrame[]

In [0]:
from pyspark.sql.functions import *

customer_features = (
    transactions
    .groupBy("card4", "card6")
    .agg(
        count("*").alias("total_transactions"),
        avg("TransactionAmt").alias("avg_transaction_amount"),
        max("TransactionAmt").alias("max_transaction_amount"),
        min("TransactionAmt").alias("min_transaction_amount"),
        expr("percentile(TransactionAmt,0.5)").alias("median_transaction_amount")
    )
)

customer_features.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("fraud_feature_store.customer_features")

In [0]:
merchant_features = (
    transactions
    .groupBy(
        "ProductCD",
        "card4",
        "card6"
    )
    .agg(
        count("*").alias("merchant_transaction_count"),
        avg("TransactionAmt").alias("merchant_avg_amount"),
        max("TransactionAmt").alias("merchant_max_amount")
    )
)

merchant_features.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("fraud_feature_store.merchant_features")

In [0]:
exchange_features = (
    exchange
    .select(
        "currency_code",
        "exchange_rate",
        "created_ts"
    )
)

In [0]:
exchange_features.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("fraud_feature_store.exchange_features")

In [0]:
ofac_features = (
    ofac
    .select(
        "entity_type",
        "entity_name",
        "country",
        "risk_score"
    )
)

In [0]:
ofac_features.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("fraud_feature_store.ofac_features")

In [0]:
tables = [
    "customer_features",
    "merchant_features",
    "country_features",
    "exchange_features",
    "ofac_features"
]

for table in tables:
    count = spark.sql(f"SELECT COUNT(*) cnt FROM fraud_feature_store.{table}").first()["cnt"]
    print(f"{table:<30} {count}")

customer_features              13553
merchant_features              269
country_features               30
exchange_features              15
ofac_features                  15


In [0]:
customer_features.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- total_transactions: long (nullable = false)
 |-- avg_transaction_amount: double (nullable = true)
 |-- max_transaction_amount: double (nullable = true)
 |-- min_transaction_amount: double (nullable = true)
 |-- median_transaction_amount: double (nullable = true)



In [0]:
merchant_features.printSchema()

root
 |-- merchant_id: string (nullable = true)
 |-- ProductCD: string (nullable = true)
 |-- card4: string (nullable = true)
 |-- card6: string (nullable = true)



In [0]:
%sql
SELECT *
FROM fraud_platform.snapshots.customer_risk_snapshot;

card4,card6,total_transactions,avg_risk_score,fraud_transactions,total_amount,dbt_scd_id,dbt_updated_at,dbt_valid_from,dbt_valid_to
discover,credit,87,33.96551724137931,12,23937.850000000002,fe1d838acd6c7809d262464ea740c4ba,2026-07-05T17:40:02.576Z,2026-07-05T17:40:02.576Z,null
american express,debit,5,15.0,0,320.0,10deeaf637481e406cf3f139205ad05c,2026-07-05T17:40:02.576Z,2026-07-05T17:40:02.576Z,null
mastercard,credit,711,32.70042194092827,41,120760.22899999989,6eff744bf6fa0bd0a3a9fa44dbf76874,2026-07-05T17:40:02.576Z,2026-07-05T17:40:02.576Z,null
mastercard,debit,2875,19.187826086956523,66,331988.1150000034,1cf0a1a1e59a87b33a8c8aea5f076e94,2026-07-05T17:40:02.576Z,2026-07-05T17:40:02.576Z,null
discover,debit,5,15.0,0,460.0,1ce0f8133ef383feea52142eed8a5e6b,2026-07-05T17:40:02.576Z,2026-07-05T17:40:02.576Z,null
american express,credit,107,21.962616822429908,3,12480.0,643a9c22b46c028bb183307ae768da20,2026-07-05T17:40:02.576Z,2026-07-05T17:40:02.576Z,null
visa,credit,1203,32.2734829592685,90,210760.4970000006,85ef3d39e5b78f250e5388821441401d,2026-07-05T17:40:02.576Z,2026-07-05T17:40:02.576Z,null
visa,debit,5006,19.58749500599281,171,614574.9609999976,1608966276ebb5c75e54957bdc66a41e,2026-07-05T17:40:02.576Z,2026-07-05T17:40:02.576Z,null
